<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">MARCO DE TRABAJO DE RECOMENDACIÓN DE TALENTO PARA LA FORMACIÓN DE EQUIPOS EN PROYECTOS DE INGENIERÍA DEL SOFTWARE</h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barceló
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 4: Evaluación — contraste con las reglas expertas)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        El objetivo de esta libreta es contrastar las <strong>reglas de asociación descubiertas</strong> en la Libreta anterior contra las <strong>27 reglas definidas por la experta de dominio</strong> Aurora Vizcaíno Barceló, para validar en qué medida los patrones encontrados en los datos coinciden, contradicen o carecen de evidencia respecto al conocimiento experto.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Carga: Primero cargamos las </strong> reglas descubiertas en el documento (<code>reglas_limpias_hard.csv</code>) y reglas expertas estructuradas, las cuales tras revisar con Aurora y Julio, las he ordenado en un archivo JSON complementado información con ayuda de Claude para poder hacer más fácil su tratamiento (<code>reglas_expertas_riesgo.json</code>).</li>
        <li><strong>Clasificación de alcance:</strong> de las 27 reglas expertas, solo una parte es evaluable con los datos de los que disponemos. Esto es porque el resto combina condiciones de nivel equipo/proyecto, que se evaluarán en fases posteriores del framework.</li>
        <li><strong>Contraste:</strong> para cada regla experta con componente de hard skills evaluable, buscamos la evidencia en las distintas reglas descubiertas y las clasificamos en base a si esta coincide, se contradice o si no tenemos evidencias suficientes.</li>
        <li><strong>Conclusiones:</strong> Estudiaremos la cobertura y los hallazgos relevantes para el objetivo central del proyecto.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 4.1. Configuración del entorno y carga de datos

En esta primera celda cargamos las reglas de asociación depuradas de la Libreta 3 (`reglas_limpias_hard.csv`, la cual ya no tienen nulos ni reglas con redundancias) y el archivo semiesctructurado con las reglas expertas (`reglas_expertas_riesgo.json`), que clasifica cada una de las 27 reglas del documento en `soft_parcial`, `hard_parcial` o `proyecto_json` según su nivel de evaluación.

In [1]:
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.hard_skills import config

reglas_limpias = pd.read_csv(config.REGLAS_LIMPIAS_PATH)
reglas_limpias['items_totales'] = reglas_limpias.apply(lambda row: set(str(row['antecedents']).split(', ')) | set(str(row['consequents']).split(', ')),axis=1)

with open(config.REGLAS_EXPERTAS_PATH, encoding='utf-8') as f:
    reglas_experto = json.load(f)['reglas']

reglas_hard = [r for r in reglas_experto if r['tipo'] == 'hard']

print(f"Reglas descubiertas en libreta 3: {len(reglas_limpias)}")
print(f"Reglas expertas totales: {len(reglas_experto)}")
print(f"Reglas expertas con componente hard evaluable: {len(reglas_hard)}")

Reglas descubiertas en libreta 3: 10107
Reglas expertas totales: 27
Reglas expertas con componente hard evaluable: 0


### 4.2. Función de Búsqueda de evidencia

Para cada las diferentes reglas de Aurora con componentes de hard skills evaluables, vamos a comprobar si sus ítems asociados **coexisten** en alguna regla descubierta (`reglas_limpias`). El motivo de este es saber si el algoritmo encontró esa combinación como suficientemente frecuente y fiable (soporte, confianza ≥0.6 ya aplicado en la Libreta 3).

In [2]:
def buscar_evidencia(items_esperados, df_reglas=reglas_limpias, top_n=5):
    items_esperados = set(items_esperados)
    masc = df_reglas['items_totales'].apply(lambda s: items_esperados.issubset(s))
    resultado = df_reglas[masc].sort_values('lift', ascending=False)
    return resultado.head(top_n)

### 4.3. Contraste de las diferentes reglas evaluables

Una vez ya tenemos todo lo necesario, para este paso vamos a recorrer las reglas de aurora con componentes de hard skills para si buscar evidencias o combinaciones de ítems con las reglas de la anterior libretas que ya tenemos limpias. A partir de ahí, a ver que obtenemos e interpretaremos los resultados manualmente más adelante.

In [3]:
for r in reglas_hard:
    print(f"\n{'='*70}")
    print(f"Regla experta #{r['id']}: {r['riesgo']}")
    print(f"Ítems localizados: {r['items_hard']}")
    if 'nota' in r:
        print(f"Nota: {r['nota']}")
    evidencia = buscar_evidencia(r['items_hard'])
    if evidencia.empty:
        print("→ SIN EVIDENCIA en reglas_limpias_hard (no coexisten con confianza≥0.6)")
    else:
        display(evidencia[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

### 4.3.1. Primera observación: 6 de 9 reglas sin evidencia directa

De las 9 reglas expertas con contenido relevante para las soft skills, vemos que en **6 de ellas no encuentran ninguna regla descubierta que las respalde** (#8, #10, #11, #22, #24, #25), y solo 3 en las que sí (#12, #23, #26).

A primera vista podemos pensar que el modelo de reglas de asociación no confirma la mayoría del conocimiento experto, pero antes de concluir con ello, vemos que hay un patrón sospechoso en las 6 sin evidencia: **`WorkRemote_Bucket_Remoto Alto (>70%)` aparece en 4 de las 6** (#11, #22, #24, #25), y las otras 2 (#8, #10) comparten `Neuroticismo_Nivel_Alto` combinado con tramos altos de `WorkWeekHrs_Bin`.

Esto no dice que la ausencia de evidencia podría no deberse a que la relación no exista en los datos, sino a una **limitación del propio proceso de minado**: